# Testing the TACK fast ensembles with AutoTPD+

Run this notebook from top to bottom with the **TACK training-image kernel**.
It downloads the public `ailab-bio/AutoTPDplus` PROTAC split, the published HF ensembles,
and their context cache. It uses no local CSV, checkpoint folder, or UniProt request.

| Task | Prediction | Units |
|---|---|---|
| `dmax` | `mean` | Dmax fraction |
| `pdc50` | `mean` | $9-\log_{10}(\mathrm{DC50\ [nM]})$ |
| `activity` | `proba` | Probability of the model's positive class |

Measured activity in this notebook means **DC50 < 100 nM and Dmax > 80%**.
A legacy classifier may use another definition; its threshold metadata is displayed when
available, and missing metadata is reported explicitly.

The examples below demonstrate inference on AutoTPD+ measurements. They are **not a held-out
performance benchmark**: some may have been used to train the published models.

1. Setup and pinned HF revisions
2. Load the three ensembles and their context
3. Inspect the input and output contracts
4. Predict AutoTPD+ measurements
5. Inspect uncertainty
6. Score several molecules in one cached biological context
7. Reproduce training and evaluation

## 1. Setup

Use `apptainer/training.sif`, which includes `datasets`, Jupyter and the inference dependencies;
see [the container instructions](../apptainer/README.md#jupyter). Select its Python kernel,
restart it, and run all cells. No package installation is performed here.

The first cell finds the checkout, loads its `.env`, and sets writable cache locations before
importing HF libraries. AutoTPD+ is public. The published model and context repositories may
require the `HF_TOKEN` from your environment or the checkout's `.env`; the token is never printed.
Once downloaded, the same pinned snapshots can be reused offline. Downloads may take several
minutes on the first run.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"  # before NumPy, XGBoost or torch
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from dotenv import load_dotenv

roots = [Path.cwd(), *Path.cwd().parents, Path("/opt/repo")]
REPO_ROOT = next((p for p in roots if (p / "tackai" / "fusion").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the TACK checkout using its training-image kernel.")
load_dotenv(REPO_ROOT / ".env")
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("HF_HOME", str(REPO_ROOT / "notebooks" / ".cache" / "hf"))
os.environ.setdefault("TACKAI_CACHE", str(REPO_ROOT / "notebooks" / ".cache" / "TACK"))
for name in ("HF_HOME", "TACKAI_CACHE"):
    Path(os.environ[name]).mkdir(parents=True, exist_ok=True)

DATASET_REPO = "ailab-bio/AutoTPDplus"
DATASET_CONFIG = "autotpdplus-protac"
DATASET_REVISION = "438177a7617386e76188e138a2c6b3da96f010f1"
MODEL_REPO = "ailab-bio/TACK-Fast-Ensemble"
MODEL_REVISION = "813f00d023b7b9e07cc57337cd0a8066e536ad2c"
CONTEXT_REPO = "ailab-bio/TACK-Context-Encoding"
LEGACY_CONTEXT_REVISION = "f3e1d9f8b4084e4af2a9f3c5cf5501f6819c3b2c"
TASKS = ("dmax", "pdc50", "activity")
N_EXAMPLES, N_SCREEN = 12, 16
OFFLINE = os.environ.get("HF_HUB_OFFLINE", "0") == "1"
if OFFLINE:
    os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Kernel:", sys.executable)
print("Measurement source:", DATASET_REPO, DATASET_CONFIG, DATASET_REVISION)
print("Model revision:", MODEL_REVISION)

## 2. Load the models and context from HF

New bundles record their exact feature layout and context revision; `load_data()` restores
those settings. The pinned legacy release lacks that metadata, so its known combined protein
space and radius-2, 4096-bit molecular layout are specified explicitly below. Every member's
input width is checked before prediction.

Context tables are read directly from the HF snapshot. This does not overwrite the shared
embedding cache or refit embeddings/PCA. All three tasks in this release share one layout.

In [ ]:
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from tackai.fusion import FusionData, XGBStackedEnsemble
from tackai.fusion.mol_encoder import DESCRIPTOR_NAMES, MolEncoder

pd.set_option("display.max_columns", 20, "display.width", 180)
ensembles = {}
for task in TASKS:
    print(f"Loading {task}…", flush=True)
    ensembles[task] = XGBStackedEnsemble.from_pretrained(
        MODEL_REPO, subfolder=task, revision=MODEL_REVISION, local_files_only=OFFLINE)
    for member in ensembles[task].models:
        member.set_params(n_jobs=1, device="cpu")

specs = [ensemble.feature_spec for ensemble in ensembles.values()]
if any(spec is not None for spec in specs):
    if any(spec != specs[0] for spec in specs):
        raise ValueError("These task bundles use different layouts; load a separate encoder for each.")
    context = ensembles["dmax"].load_data(local_files_only=OFFLINE)
else:
    context_snapshot = snapshot_download(
        CONTEXT_REPO, repo_type="dataset", revision=LEGACY_CONTEXT_REVISION,
        local_files_only=OFFLINE)
    context = FusionData.from_pretrained(
        context_snapshot, cache_dir=context_snapshot, protein_space="combined",
        featurizer=MolEncoder(radius=2, fp_size=4096, descriptors=DESCRIPTOR_NAMES))

for task, ensemble in ensembles.items():
    widths = {int(member.n_features_in_) for member in ensemble.models}
    if widths != {context.n_columns}:
        raise ValueError(f"{task}: model widths {widths} differ from the encoder's {context.n_columns}.")
    if ensemble.target_name not in (None, task):
        raise ValueError(f"{task}: bundle target is {ensemble.target_name!r}.")

layout = pd.DataFrame([{"block": block, "columns": len(cols)}
                       for block, cols in context.blocks_indexes.items()])
display(layout)
display(pd.DataFrame([{
    "task": task, "members": len(ensemble.models), "family": ensemble.task,
    "feature_metadata": ensemble.feature_spec is not None,
} for task, ensemble in ensembles.items()]))
print("Total columns:", context.n_columns)
print("Activity threshold metadata:",
      (ensembles["activity"].feature_spec or {}).get("activity_thresholds", "not supplied by this legacy release"))

## 3. Inspect inputs and outputs

A record supplies a molecule plus its biological context. The context cache already contains
the protein sequences, cell-line keys and assay representations used for this release.
`help()` exposes the contract without looking at source code.

In [ ]:
def show(obj, indent=0):
    """Print nested API help without expanding every cached sequence."""
    pad = "  " * indent
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict):
                print(f"{pad}{key}:")
                show(value, indent + 1)
            else:
                print(f"{pad}{key}: {value}")
    else:
        print(f"{pad}{obj}")

show(context.help())

Protein sequences and nonblank cell-line identifiers must be encodable by the published cache.
An absent protein is not silently replaced. Missing cells use the unknown-cell vector;
unrecognized assay text follows the encoder's documented normalization/fallback rules.

The examples will be selected from compatible AutoTPD+ rows, and their sequences come from
that dataset directly. No sequence is fetched from another service.

In [ ]:
for block in ("cell_id", "poi_seq", "e3_seq"):
    print(f"{block:<9} {context.help(block=block)['n_keys']:>6} cached keys")
show(context.help(block="assay", return_keys=True))

## 4. Load and predict AutoTPD+ measurements

The pinned public PROTAC split is harmonized with the same `build_table()` function used by
training. We select uncensored measurements with both endpoints, matching recorded DC50/Dmax
treatment durations and encodable context, then spread the examples across the observed DC50 range. Counts make the filtering
visible. This selection is for a compact inference demonstration, not model selection or
an estimate of generalization error.

In [ ]:
from datasets import load_dataset, disable_progress_bars
from rdkit import Chem, RDLogger
from tackai.fusion.data import build_table

disable_progress_bars()
RDLogger.DisableLog("rdApp.*")
source = load_dataset(DATASET_REPO, DATASET_CONFIG, revision=DATASET_REVISION)["train"].to_pandas()
source.attrs["name"] = DATASET_CONFIG
same_time = source.DC50_h.notna() & source.Dmax_h.notna() & source.DC50_h.eq(source.Dmax_h)
exact = np.ones(len(source), dtype=bool)
for column in ("DC50_operator", "Dmax_operator"):
    exact &= source[column].fillna("=").astype(str).str.strip().isin(["", "=", "=="])
demo_source = source.loc[same_time & exact].copy()
demo_source.attrs["name"] = DATASET_CONFIG
table = build_table([demo_source])
keep = np.isfinite(table.dmax_pct) & np.isfinite(table.dc50_nM) & (table.dc50_nM > 0)
keep &= table.assay_time.notna() & table.cell_key.notna() & table.assay_raw.notna()
for block, column in (("poi", "poi_seq"), ("e3", "e3_seq"), ("cell", "cell_key")):
    keep &= context.encoder.encodable(block, table[column])
keep &= table.smiles.map(lambda s: Chem.MolFromSmiles(str(s)) is not None)
context_columns = ["poi_seq", "e3_seq", "cell_key", "assay_raw", "assay_time"]
candidates = table.loc[keep].drop_duplicates(["smiles", *context_columns]).copy()
if candidates.empty:
    raise RuntimeError("No compatible AutoTPD+ measurements for this context snapshot.")
ordered = candidates.sort_values(["dc50_nM", "dmax_pct"], kind="stable")
positions = np.linspace(0, len(ordered) - 1, min(N_EXAMPLES, len(ordered)), dtype=int)
benchmark = ordered.iloc[positions].reset_index(drop=True)
benchmark.insert(0, "example", np.arange(1, len(benchmark) + 1))
print(f"HF rows: {len(source):,}; uncensored rows with matched duration and SMILES: {len(table):,}; compatible paired measurements: {len(candidates):,}")
display(benchmark[["example", "poi_uniprot", "e3_raw", "cell_key", "assay_raw",
                   "assay_time", "dc50_nM", "dmax_pct"]])

### Encode records using the sequences supplied by AutoTPD+

DC50 is in nM and Dmax in percent in the source. Context input uses full amino-acid sequences,
cell-line keys, assay text and time in hours. The function below is reused for the screening
example so there is one record-construction path.

In [ ]:
def records_from_frame(frame):
    """Convert the harmonized AutoTPD+ rows into the public inference contract."""
    return [{
        "smiles": row.smiles, "poi_seq": row.poi_seq, "e3_seq": row.e3_seq,
        "cell_id": row.cell_key, "assay": row.assay_raw,
        "assay_time": float(row.assay_time),
    } for row in frame.itertuples()]

records = records_from_frame(benchmark)
X = context.transform(records)
assert X.shape == (len(benchmark), context.n_columns)
print("Encoded examples:", X.shape)

### Predict all three tasks

Measured activity uses the requested strict 100-nM / 80% thresholds. The model probability
is shown separately because a legacy release does not serialize its positive-class threshold.
Do not interpret an unknown-threshold classifier as a calibrated 100-nM activity model.

In [ ]:
from tackai.fusion.data import make_targets, DMAX_THR, PDC50_THR

assert (DMAX_THR, PDC50_THR) == (0.8, 7.0)
measured = make_targets(benchmark.dmax_pct, benchmark.dc50_nM)
predicted = {task: ensembles[task].predict(X) for task in TASKS}
for task, outputs in predicted.items():
    key = "proba" if task == "activity" else "mean"
    assert outputs[key].shape == (len(benchmark),)
    assert np.isfinite(outputs[key]).all(), f"Nonfinite {task} point predictions"
assert ((predicted["activity"]["proba"] >= 0) & (predicted["activity"]["proba"] <= 1)).all()
results = pd.DataFrame({
    "example": benchmark.example, "POI": benchmark.poi_uniprot, "cell": benchmark.cell_key,
    "Dmax": measured["dmax"], "Dmax_pred": predicted["dmax"]["mean"],
    "Dmax_std": predicted["dmax"]["std"], "pDC50": measured["pdc50"],
    "pDC50_pred": predicted["pdc50"]["mean"], "pDC50_std": predicted["pdc50"]["std"],
    "active_100nM": measured["activity"], "P_active_model": predicted["activity"]["proba"],
})
display(results.round(3))

## 5. Inspect uncertainty

Regression separates model disagreement from fitted noise. A calibrated regression ensemble
also provides `lower` and `upper`; the classification ensemble reports entropy terms.
`help(show_extras=True)` explains the actual keys and parameters of this downloaded release.
These quantities are diagnostics; low disagreement does not prove that a new context is in
training distribution.

In [ ]:
for task in TASKS:
    print(f"\n{task}")
    show(ensembles[task].help(show_extras=True))

Review the examples with the largest Dmax uncertainty. Conformal coverage depends on the
calibration assumptions and can deteriorate under distribution shift. This notebook does not
fit weights, temperatures or intervals on the example measurements.

In [ ]:
flagged = results.assign(uncertain=results.Dmax_std > results.Dmax_std.median())
display(flagged[["example", "POI", "Dmax", "Dmax_pred", "Dmax_std",
                 "P_active_model", "uncertain"]].sort_values("Dmax_std", ascending=False).round(3))

## 6. Screen several molecules in one cached biological context

Choose the AutoTPD+ context with the most distinct compatible molecules. These measurements
share target sequence, E3 sequence, cell line, assay and duration. Encode that context once,
then assemble each molecular vector with it. This demonstrates the screening API without a
hardcoded literature series or an external sequence lookup.

In [ ]:
grouped = candidates.groupby(context_columns, sort=False, dropna=False)
chosen_context = grouped.smiles.nunique().idxmax()
series_pool = grouped.get_group(chosen_context).drop_duplicates("smiles").sort_values("dc50_nM")
positions = np.linspace(0, len(series_pool) - 1, min(N_SCREEN, len(series_pool)), dtype=int)
series = series_pool.iloc[positions].reset_index(drop=True)
series_records = records_from_frame(series)
shared_context = {key: value for key, value in series_records[0].items() if key != "smiles"}
encoded_context = context.transform_context(shared_context)
X_series = context.assemble(series.smiles.tolist(), encoded_context)
np.testing.assert_array_equal(X_series, context.transform(series_records))
truth = make_targets(series.dmax_pct, series.dc50_nM)
series_predictions = {task: ensembles[task].predict(X_series) for task in TASKS}
ranked = pd.DataFrame({
    "example": np.arange(1, len(series) + 1), "pDC50": truth["pdc50"],
    "pDC50_pred": series_predictions["pdc50"]["mean"], "Dmax": truth["dmax"],
    "Dmax_pred": series_predictions["dmax"]["mean"],
    "P_active_model": series_predictions["activity"]["proba"],
}).sort_values("pDC50_pred", ascending=False)
print("Shared context:", series.iloc[0][["poi_uniprot", "e3_raw", "cell_key", "assay_raw", "assay_time"]].to_dict())
display(ranked.round(3))

Spearman correlation below describes ordering within these displayed AutoTPD+ examples.
It is not an independent screening benchmark or evidence that the models generalize to a new
chemical series. A constant measured or predicted series has undefined correlation.

In [ ]:
from scipy.stats import spearmanr

for task, column in (("pdc50", "pDC50"), ("dmax", "Dmax")):
    if ranked[column].nunique() > 1 and ranked[f"{column}_pred"].nunique() > 1:
        rho = float(spearmanr(ranked[column], ranked[f"{column}_pred"]).statistic)
        print(f"{task:<6} descriptive Spearman rho = {rho:+.3f}")
    else:
        print(f"{task:<6} descriptive Spearman rho = undefined (constant series)")

## 7. Reproduce training and evaluation

- Measurement rows in this notebook come only from **AutoTPDplus** at the pinned revision.
- Change the HF repository/revision constants in the setup cell to test a newly published
  ensemble release. New bundles carry their target, context revision, feature layout,
  activity thresholds and training provenance; the loader restores them automatically.
- Molecular geometry and descriptor selection belong to each model bundle. A legacy bundle
  without metadata needs the explicit release-specific fallback shown above.
- New training uses activity defined by DC50 < 100 nM and Dmax > 80%. The downloaded legacy
  classifier's missing threshold metadata remains visible rather than being guessed.
- Embeddings and PCA are loaded from HF and never regenerated in this notebook. Protein
  sequences come directly from AutoTPD+, so a UniProt release change cannot break the example.
- For honest performance comparisons, use the fixed held-out matrices and grouped CV in
  [the reproduction guide](../REPRODUCING.md), not these demonstration rows. See
  [the model audit](../docs/FAST_MODELS_DEBUG.md) for measured results and remaining limitations.

The notebook performs inference only and creates no Hub repository or upload.